<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_05_classification/05_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 5 folder
%cd chapter_05_classification

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 184 (delta 111), reused 70 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 614.34 KiB | 3.89 MiB/s, done.
Resolving deltas: 100% (111/111), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_05_classification


# Chapter 05: Classification

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 5: Classification

## Goals

In this notebook, I will learn:
- Classification vs regression
- Binary classification
- Logistic regression and probabilities
- Decision thresholds
- Confusion matrix
- Precision, recall, F1-score
- ROC and AUC
- Naive Bayes
- Linear Discriminant Analysis (LDA)
- K-nearest neighbours (KNN)
- Class imbalance and strategies to handle it
- Odds ratios and model interpretation

---

## 🎯 Learning Objectives

1. Apply Naive Bayes for categorical and numeric predictors
2. Understand Linear Discriminant Analysis (LDA) and covariance
3. Fit and interpret Logistic Regression models (odds ratios, logit)
4. Evaluate classifiers beyond accuracy: Precision, Recall, Specificity, ROC, and AUC
5. Address the Rare Class Problem using undersampling, oversampling, and class weighting

---

## 🤖 ML Relevance

> Why does classification matter for Machine Learning?
> * **Core Business Logic**: Most automated business decisions (churn prediction, fraud detection, click-through rates, loan defaults) are binary classification problems.
> * **Probability vs. Hard Labels**: In production, we rarely use the hard 0/1 cutoff. We use the *propensity score* (probability) to rank users, allocate resources, or calculate Expected Value.
> * **The Imbalanced Data Trap**: Real-world positive events (fraud, conversion) are rare. Standard algorithms will simply predict "0" every time and achieve 99% accuracy while being completely useless. Knowing how to handle this is a senior-level DS skill.

---

## Setup & Data Preparation

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             precision_score, recall_score, f1_score, accuracy_score,
                             roc_curve, roc_auc_score, auc)

# Helper to load book datasets
def load_data(filename, url):
    local_path = f'../datasets/raw/{filename}'
    if os.path.exists(local_path):
        return pd.read_csv(local_path)
    else:
        print(f"Downloading {filename}...")
        return pd.read_csv(url)

# Load Loan Data
loan_url = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/loan_data.csv"
loan3000_url = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/loan3000.csv"

loan_data = load_data('loan_data.csv', loan_url)
loan3000 = load_data('loan3000.csv', loan3000_url)

# Ensure categorical columns are properly typed
for col in ['outcome', 'purpose_', 'home_', 'emp_len_']:
    if col in loan_data.columns:
        loan_data[col] = loan_data[col].astype('category')
    if col in loan3000.columns:
        loan3000[col] = loan3000[col].astype('category')

print("Datasets loaded successfully.")
print(f"Full Loan Data shape: {loan_data.shape}")
print(f"Loan 3000 subset shape: {loan3000.shape}")
print(f"\nTarget Distribution (Full Data):")
print(loan_data['outcome'].value_counts(normalize=True))
```
</details>



---

# Section 1: Classification vs Regression

## Key Difference

- **Regression** predicts continuous numbers (house price, salary, temperature)
- **Classification** predicts categories (spam/not spam, fraud/not fraud, delayed/on-time)

## Example Scenario

We want to predict flight delays:
- **0**: On-time
- **1**: Delayed

<details>
<summary>Click to view solution</summary>

```python
# Simulated airline dataset
np.random.seed(42)

airline_data = pd.DataFrame({
    "Distance": np.random.normal(1000, 300, 1000),
    "Weather_Score": np.random.normal(5, 2, 1000),
    "Airport_Congestion": np.random.normal(50, 15, 1000)
})

delay_probability = (
    0.002 * airline_data["Distance"] +
    0.08 * airline_data["Weather_Score"] +
    0.01 * airline_data["Airport_Congestion"]
)

airline_data["Delayed"] = (
    np.random.rand(1000) < (delay_probability / delay_probability.max())
).astype(int)

print(airline_data.head())
print("\nTarget distribution:")
print(airline_data["Delayed"].value_counts())
```
</details>

## Reflection

Questions:
1. What are the features?
2. What is the target variable?
3. Why is this classification and not regression?


---

# Section 2: Binary Classification

## What is binary classification?

Binary classification predicts two classes (0/1, Yes/No, Spam/Not Spam).

<details>
<summary>Click to view solution</summary>

```python
sns.countplot(data=airline_data, x="Delayed")
plt.title("Flight Delay Distribution")
plt.show()
```
</details>

## Reflection

Questions:
1. Which class appears more often?
2. Is dataset balanced?
3. Why might imbalance matter?



---

# Section 3: Logistic Regression

## Why not linear regression?

Linear regression can predict negative probabilities or values > 1. Logistic regression predicts probability between 0 and 1.

## Logistic Regression Output

Model learns probability of belonging to a class. Example: 0.95 means 95% confidence for positive class.

<details>
<summary>Click to view solution</summary>

```python
X = airline_data[["Distance", "Weather_Score", "Airport_Congestion"]]
y = airline_data["Delayed"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"Coefficients: {model.coef_[0]}")
print(f"Intercept: {model.intercept_[0]:.4f}")
```
</details>

## Reflection

Questions:
1. Was accuracy high?
2. Does high accuracy always mean good model?
3. What problems could accuracy hide?


---

# Section 4: Probabilities

Instead of hard prediction (0 or 1), the model first predicts probabilities.

<details>
<summary>Click to view solution</summary>

```python
probabilities = model.predict_proba(X_test)

probability_df = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": predictions[:10],
    "Delay Probability": probabilities[:10, 1]
})
print(probability_df)
print("\nInterpretation: [P(not delayed), P(delayed)]")
print("Example: [0.20, 0.80] means 20% chance not delayed, 80% chance delayed.")
```
</details>

## Reflection

Questions:
1. Why are probabilities useful?
2. Why not directly predict classes?
3. When might uncertainty matter?



---

# Section 5: Decision Thresholds

Default threshold is 0.50. If probability > 0.50, predict 1. Otherwise, predict 0.

<details>
<summary>Click to view solution</summary>

```python
# Custom threshold experiment
custom_predictions = (probabilities[:, 1] > 0.70).astype(int)
custom_accuracy = accuracy_score(y_test, custom_predictions)

print(f"Accuracy with 50% threshold: {accuracy:.4f}")
print(f"Accuracy with 70% threshold: {custom_accuracy:.4f}")
print("\nChanging threshold affects precision/recall trade-off.")
```
</details>

## Reflection

Questions:
1. What changed after threshold adjustment?
2. Why might business change threshold?
3. Should fraud detection use 50% threshold?



---

# Section 6: Logistic Regression with Loan Data (Interpretation Focus)

## Fitting and Interpreting via Statsmodels

Scikit-learn is great for prediction, but `statsmodels` gives us p-values, standard errors, and Odds Ratios.

<details>
<summary>Click to view solution</summary>

```python
# Prepare data
logit_data = loan_data.dropna(subset=['payment_inc_ratio', 'purpose_', 'home_', 'emp_len_', 'borrower_score', 'outcome'])
logit_data['outcome_binary'] = (logit_data['outcome'] == 'default').astype(int)

formula = 'outcome_binary ~ payment_inc_ratio + purpose_ + home_ + emp_len_ + borrower_score'

model_logit = smf.glm(formula=formula, data=logit_data,
                      family=sm.families.Binomial()).fit()

# Extract Odds Ratios
params = model_logit.params
odds_ratios = np.exp(params)

results_df = pd.DataFrame({
    'Coefficient': params,
    'Odds Ratio': odds_ratios,
    'P-value': model_logit.pvalues
})
print("Logistic Regression Results (Exponentiated to Odds Ratios):")
print(results_df.round(4))
print("\nInterpretation: Odds Ratio > 1 means predictor increases odds of default.")
print("Odds Ratio < 1 means predictor decreases odds of default.")
```
</details>



---

# Section 7: Confusion Matrix

## What is a confusion matrix?

A confusion matrix helps us understand where model succeeds and fails by comparing actual vs predicted.

### Four Outcomes
- **True Positive (TP)**: Predicted Delayed, Actual Delayed
- **True Negative (TN)**: Predicted On-time, Actual On-time
- **False Positive (FP)**: Predicted Delayed, Actual On-time
- **False Negative (FN)**: Predicted On-time, Actual Delayed

<details>
<summary>Click to view solution</summary>

```python
cm = confusion_matrix(y_test, predictions)
print("Confusion Matrix:")
print(cm)

display = ConfusionMatrixDisplay(confusion_matrix=cm)
display.plot()
plt.title("Confusion Matrix")
plt.show()
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Confusion Matrix for loan data logistic regression
X_logit = pd.get_dummies(logit_data[['payment_inc_ratio', 'purpose_', 'home_', 'emp_len_', 'borrower_score']], drop_first=True)
y_logit = logit_data['outcome_binary']

sk_logit = LogisticRegression(penalty='l2', C=1e42, solver='liblinear', random_state=42)
sk_logit.fit(X_logit, y_logit)

y_pred_loan = sk_logit.predict(X_logit)
y_prob_loan = sk_logit.predict_proba(X_logit)[:, 1]

cm_loan = confusion_matrix(y_logit, y_pred_loan)
print("Confusion Matrix (Loan Data):")
print(pd.DataFrame(cm_loan, index=['Actual Paid', 'Actual Default'],
                   columns=['Pred Paid', 'Pred Default']))
```
</details>

## Reflection

Questions:
1. Which mistake happened more?
2. Why are false negatives dangerous?
3. Why are false positives costly?



---

# Section 8: Accuracy and Its Limitations

Accuracy measures overall correctness: correct predictions / total predictions.

⚠️ **Important**: High accuracy can be misleading, especially for imbalanced datasets.

Example: If 99% flights are on-time, a model always predicting "on-time" has 99% accuracy but is useless.


---

# Section 9: Precision

## What is precision?

When model predicts positive, how often is it correct?

Formula: TP / (TP + FP)

**When precision matters**: Spam detection, fraud detection, loan approval (false positives are expensive).

<details>
<summary>Click to view solution</summary>

```python
precision = precision_score(y_test, predictions)
print(f"Precision: {precision:.4f}")
print("Of all flights predicted as delayed, this fraction actually were delayed.")
```
</details>



---

# Section 10: Recall

## What is recall?

How many true positives did model find?

Formula: TP / (TP + FN)

**When recall matters**: Medical diagnosis, fraud detection, terror detection (missing positives is dangerous).

<details>
<summary>Click to view solution</summary>

```python
recall = recall_score(y_test, predictions)
print(f"Recall: {recall:.4f}")
print("Of all actually delayed flights, this fraction were correctly identified.")
```
</details>


---

# Section 11: F1 Score

F1 balances precision and recall. Useful when class imbalance exists.

<details>
<summary>Click to view solution</summary>

```python
f1 = f1_score(y_test, predictions)
print(f"F1 Score: {f1:.4f}")
```
</details>

## Reflection

Questions:
1. Which metric matters most?
2. Would airline delays prioritise recall?
3. Would fraud detection prioritise precision?



---

# Section 12: Classification Report

<details>
<summary>Click to view solution</summary>

```python
print("Classification Report:")
print(classification_report(y_test, predictions))
```
</details>

## Reflection

Questions:
1. Which metric looked strongest?
2. Which class performed poorly?
3. Why analyse more than accuracy?


---

# Section 13: ROC Curve and AUC

## What is ROC?

ROC measures trade-off between True Positive Rate (Recall) and False Positive Rate.

Closer to top-left means better classifier.

## What is AUC?

AUC = Area Under Curve. Range: 0.5 (random guessing) to 1.0 (perfect classifier).

<details>
<summary>Click to view solution</summary>

```python
# ROC Curve for airline data
fpr, tpr, thresholds_roc = roc_curve(y_test, probabilities[:, 1])
auc_score = roc_auc_score(y_test, probabilities[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], linestyle="--", label='Random Guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"AUC Score: {auc_score:.4f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# ROC Curve for loan data
fpr_loan, tpr_loan, thresholds_loan = roc_curve(y_logit, y_prob_loan)
roc_auc_loan = auc(fpr_loan, tpr_loan)

plt.figure(figsize=(7, 6))
plt.plot(fpr_loan, tpr_loan, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_loan:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall / Sensitivity)')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()
```
</details>

## Reflection

Questions:
1. Was AUC high?
2. Why is AUC useful?
3. Why better than accuracy?


---

# Section 14: Threshold Experiment

<details>
<summary>Click to view solution</summary>

```python
thresholds_exp = [0.30, 0.50, 0.70, 0.90]

for threshold in thresholds_exp:
    custom_predictions = (probabilities[:, 1] > threshold).astype(int)
    print(f"\nThreshold: {threshold}")
    print(f"  Precision: {precision_score(y_test, custom_predictions, zero_division=0):.4f}")
    print(f"  Recall: {recall_score(y_test, custom_predictions, zero_division=0):.4f}")
    print(f"  Accuracy: {accuracy_score(y_test, custom_predictions):.4f}")
```
</details>

## Reflection

Questions:
1. What happened to recall as threshold increased?
2. What happened to precision?
3. Why are thresholds business-dependent?


---

# Section 15: Class Imbalance

## What is class imbalance?

When one class dominates (e.g., 99% non-fraud, 1% fraud). Accuracy becomes misleading.

<details>
<summary>Click to view solution</summary>

```python
imbalanced_target = np.random.choice([0, 1], size=1000, p=[0.95, 0.05])
print("Imbalanced distribution:")
print(pd.Series(imbalanced_target).value_counts())

sns.countplot(x=imbalanced_target)
plt.title("Imbalanced Dataset (95% class 0, 5% class 1)")
plt.show()
```
</details>

## Reflection

Questions:
1. Why is imbalance difficult?
2. Why can accuracy fail?
3. Which metrics matter more?



---

# Section 16: Strategies for Imbalanced Data

## Solutions
1. **Undersampling**: Randomly drop majority class records
2. **Oversampling / SMOTE**: Duplicate or synthetically generate minority class records
3. **Class Weighting**: Penalize mistakes on minority class more heavily

<details>
<summary>Click to view solution</summary>

```python
# Compare Standard vs Balanced Logistic Regression
X_full = pd.get_dummies(loan_data[['payment_inc_ratio', 'purpose_', 'home_', 'emp_len_', 'borrower_score']].dropna(), drop_first=True)
y_full = (loan_data.loc[X_full.index, 'outcome'] == 'default').astype(int)

# Standard Model
logit_standard = LogisticRegression(penalty='l2', C=1e42, solver='liblinear', random_state=42)
logit_standard.fit(X_full, y_full)
pred_standard = logit_standard.predict(X_full)

# Balanced Model
logit_balanced = LogisticRegression(penalty='l2', C=1e42, solver='liblinear', class_weight='balanced', random_state=42)
logit_balanced.fit(X_full, y_full)
pred_balanced = logit_balanced.predict(X_full)

print("Standard Model Predictions Distribution:")
print(pd.Series(pred_standard).value_counts(normalize=True))
print(f"Recall (Default): {recall_score(y_full, pred_standard):.4f}\n")

print("Balanced Model Predictions Distribution:")
print(pd.Series(pred_balanced).value_counts(normalize=True))
print(f"Recall (Default): {recall_score(y_full, pred_balanced):.4f}")

print("\nInsight: Balanced model catches far more actual defaults (higher Recall)")
print("at the cost of more False Positives.")
```
</details>



---

# Section 17: Naive Bayes

## What is Naive Bayes?

A probabilistic classifier based on Bayes' theorem. Called "naive" because it assumes features are independent. Used often for spam detection, document classification, sentiment analysis.

<details>
<summary>Click to view solution</summary>

```python
# Multinomial Naive Bayes for loan data
predictors_nb = ['purpose_', 'home_', 'emp_len_']
outcome_nb = 'outcome'

X_nb = pd.get_dummies(loan_data[predictors_nb], prefix='', prefix_sep='')
y_nb = loan_data[outcome_nb]

X_train_nb, X_test_nb, y_train_nb, y_test_nb = train_test_split(X_nb, y_nb, test_size=0.2, random_state=42)

nb_model = MultinomialNB(alpha=0.01, fit_prior=True)
nb_model.fit(X_train_nb, y_train_nb)

nb_preds = nb_model.predict(X_test_nb)

print("Naive Bayes Classification Report:")
print(classification_report(y_test_nb, nb_preds))
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Gaussian Naive Bayes for airline data
nb_model_gaussian = GaussianNB()
nb_model_gaussian.fit(X_train, y_train)
nb_predictions = nb_model_gaussian.predict(X_test)

print(f"Naive Bayes Accuracy: {accuracy_score(y_test, nb_predictions):.4f}")
```
</details>

## Reflection

Questions:
1. How did Naive Bayes perform?
2. Why might independence assumption fail?
3. Why is Naive Bayes still useful despite its simplicity?



---

# Section 18: Linear Discriminant Analysis (LDA)

LDA classifies data by finding a linear combination of features that maximizes separation between classes. It assumes predictors are normally distributed and all classes share the same covariance matrix.

<details>
<summary>Click to view solution</summary>

```python
# LDA on numeric predictors
predictors_lda = ['borrower_score', 'payment_inc_ratio']
X_lda = loan3000[predictors_lda]
y_lda = loan3000['outcome']

lda = LinearDiscriminantAnalysis()
lda.fit(X_lda, y_lda)

lda_probs = pd.DataFrame(lda.predict_proba(X_lda), columns=lda.classes_)
lda_df = pd.concat([loan3000, lda_probs[['default']]], axis=1)

# Visualize Decision Boundary
plt.figure(figsize=(8, 6))
sns.scatterplot(x='borrower_score', y='payment_inc_ratio', hue='default',
                data=lda_df, palette='coolwarm', alpha=0.6, legend=False)

center = np.mean(lda.means_, axis=0)
slope = -lda.scalings_[0, 0] / lda.scalings_[1, 0]
intercept = center[1] - center[0] * slope
x_vals = np.array([0.15, 0.80])
y_vals = intercept + slope * x_vals

plt.plot(x_vals, y_vals, color='darkgreen', linewidth=3, label='LDA Decision Boundary')
plt.xlim(0.15, 0.80)
plt.ylim(0, 20)
plt.title('LDA Prediction: Loan Default Probability')
plt.xlabel('Borrower Score')
plt.ylabel('Payment-to-Income Ratio')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```
</details>


---

# Section 19: K-Nearest Neighbours (KNN)

## What is KNN?

KNN predicts class using nearest neighbours. Similar examples have similar labels.

<details>
<summary>Click to view solution</summary>

```python
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

knn_predictions = knn_model.predict(X_test)
print(f"KNN Accuracy: {accuracy_score(y_test, knn_predictions):.4f}")
```
</details>

## Choosing K

Small K → sensitive to noise. Large K → too smooth.

<details>
<summary>Click to view solution</summary>

```python
k_values = [1, 3, 5, 10, 20]
accuracies_knn = []

for k in k_values:
    model_k = KNeighborsClassifier(n_neighbors=k)
    model_k.fit(X_train, y_train)
    preds_k = model_k.predict(X_test)
    accuracies_knn.append(accuracy_score(y_test, preds_k))

plt.figure(figsize=(8, 5))
plt.plot(k_values, accuracies_knn, marker="o", linewidth=2)
plt.xlabel("K (Number of Neighbors)")
plt.ylabel("Accuracy")
plt.title("K Value vs Accuracy")
plt.grid(alpha=0.3)
plt.show()
```
</details>

## Reflection

Questions:
1. Which K performed best?
2. Why can very small K overfit?
3. Why can large K underfit?


---

# Section 20: Comparing Models

<details>
<summary>Click to view solution</summary>

```python
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "KNN"],
    "Accuracy": [
        accuracy_score(y_test, predictions),
        accuracy_score(y_test, nb_predictions),
        accuracy_score(y_test, knn_predictions)
    ]
})
print(comparison)

sns.barplot(data=comparison, x="Model", y="Accuracy")
plt.title("Model Comparison")
plt.xticks(rotation=45)
plt.show()
```
</details>

## Reflection

Questions:
1. Which model performed best?
2. Does best accuracy mean best model?
3. What trade-offs exist (speed, interpretability, assumptions)?



---

# ML Relevance

## Why Chapter 5 matters in ML

Classification powers real-world ML systems:

### Aviation
- Delay prediction, customer complaint prediction, fraud detection

### Finance
- Loan approval, credit risk, fraud detection

### Healthcare
- Disease prediction, medical diagnosis

### Technology
- Spam filtering, recommendation systems, content moderation

### Key ML Ideas
- Precision, recall, threshold tuning
- Class imbalance handling
- Probability vs hard labels
- ROC-AUC for model comparison

### ML Relevance Recap

| Concept | ML Application |
|---------|---------------|
| **Propensity Scores** | Ranking users for marketing campaigns rather than hard-classifying |
| **Odds Ratios** | Explaining why a model denied a loan (Explainable AI / Regulatory compliance) |
| **ROC / AUC** | Comparing rank-ordering power of Model A vs Model B |
| **Class Weights** | Simplest way to handle fraud/anomaly detection |

---

# Mini Project

## Goal

Build your own classification system. Choose one:
1. Flight delay prediction
2. Loan approval
3. Customer churn
4. Spam detection
5. Fraud detection

### Tasks
1. Create dataset
2. Train classifier
3. Compare 2-3 models
4. Evaluate metrics (precision, recall, F1, AUC)
5. Tune threshold
6. Explain business meaning



---

# Interview Questions

1. **Difference between regression and classification?**
2. **Why use logistic regression?**
3. **What is precision?**
4. **What is recall?**
5. **Difference between precision and recall?**
6. **What is F1-score?**
7. **Why can accuracy be misleading?**
8. **What is class imbalance?**
9. **Difference between KNN and logistic regression?**
10. **Why use ROC-AUC?**
11. **What are odds ratios and how do you interpret them?**
12. **How does Naive Bayes work and why is it called "naive"?**
13. **What strategies exist for handling imbalanced data?**

---

# Chapter Summary

## Key Concepts Learned
- Classification vs regression
- Logistic regression and probabilities
- Decision thresholds
- Confusion matrix (TP, TN, FP, FN)
- Precision, recall, F1-score
- ROC curve and AUC
- Naive Bayes and LDA
- K-Nearest Neighbours
- Class imbalance strategies (class weighting, oversampling, undersampling)
- Odds ratios for model interpretation

## Biggest takeaway

Classification is about making decisions under uncertainty. Good models optimise trade-offs (precision vs recall), not simply accuracy. When dealing with rare events, never trust accuracy alone—use precision, recall, F1, and AUC.

---

# Progress Checklist

- [ ] Understood classification vs regression
- [ ] Used logistic regression
- [ ] Understood probabilities and thresholds
- [ ] Interpreted odds ratios
- [ ] Used confusion matrix
- [ ] Learned precision, recall, F1-score
- [ ] Understood ROC-AUC
- [ ] Used Naive Bayes (Multinomial and Gaussian)
- [ ] Applied Linear Discriminant Analysis
- [ ] Used KNN and tuned K value
- [ ] Understood class imbalance
- [ ] Applied class weighting for imbalanced data
- [ ] Compared multiple models
- [ ] Connected classification to ML
- [ ] Ready for Chapter 6

---

> **Pro Tip**: In the real world, stakeholders will ask you to "optimize for accuracy." It is your job as a Data Scientist to push back, explain the Rare Class Problem, and optimize for Recall, Precision, or Expected Profit instead. The `class_weight='balanced'` parameter in Scikit-learn is often the simplest and most effective first step.
